In [ ]:
import os
import sys
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import DataLoader, Subset
sys.path.append(os.path.abspath(".."))
from src.data.dataset import INRDataset
from notebooks.generative import INRImageGenerator
from src.utils.prepare_data import INRDataProcessor
from src.models.gpt import INRGPT

class YamlConfig:
    def __init__(self, config_dict):
        for key, value in config_dict.items():
            setattr(self, key, value)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using target hardware acceleration: {device}")

if torch.cuda.is_available():
    torch.set_float32_matmul_precision("high")

# --- Load INRGPT Checkpoint ---
CHECKPOINT_PATH = "../checkpoints/inr_gpt_final.pt"
if not os.path.exists(CHECKPOINT_PATH):
    raise FileNotFoundError(f"Missing checkpoint file at {CHECKPOINT_PATH}")

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
config = checkpoint.get("config")

print("Loading trained INRGPT engine...")
model = INRGPT(config).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

LENET_CHECKPOINT_PATH = "../MNIST-classifier/lenet5_mnist.pth" 

print("Loading pretrained LeNet metric engine...")

class LeNet5(nn.Module):
    def __init__(self):
        super(LeNet5, self).__init__()
        # Feature Extractor
        self.feature_extractor = nn.Sequential(            
            nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5, stride=1), # Output: 24x24
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),                             # Output: 12x12
            
            nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5, stride=1), # Output: 8x8
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)                              # Output: 4x4
        )
        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(in_features=16 * 4 * 4, out_features=120),               # Changed from 16*5*5 to 16*4*4
            nn.ReLU(),
            nn.Linear(in_features=120, out_features=84),
            nn.ReLU(),
            nn.Linear(in_features=84, out_features=10)
        )

    def forward(self, x):
        x = self.feature_extractor(x)
        x = torch.flatten(x, 1) 
        logits = self.classifier(x)
        return logits

lenet_model = LeNet5().to(device)
lenet_model.load_state_dict(torch.load(LENET_CHECKPOINT_PATH, map_location=device)['model_state_dict'])
lenet_model.eval()

print("Loading dataset for structural template allocation...")
full_dataset = INRDataset(folder_path="../data/processed_inrs", split="train")

template_loader = DataLoader(
    Subset(full_dataset, range(64)),
    batch_size=64,
    shuffle=False,
    pin_memory=(device == "cuda"),
    num_workers=1,
)

reference_batch = next(iter(template_loader))
processor = INRDataProcessor(input_root="../data/mnist-inrs-relus/all_inrs_eight_neurons")

print(f"✅ Structural template locked. Sequence mapping layout shape: {reference_batch['layer_ids'][0].shape}")

def calculate_lenet_score(probs, splits=10):
    """
    Computes the Inception Score math variant using LeNet probability distributions.
    """
    N = probs.size(0)
    split_scores = []

    for i in range(splits):
        part = probs[i * (N // splits): (i + 1) * (N // splits)]
        # Marginal distribution p(y) over the current split chunk
        p_y = torch.mean(part, dim=0, keepdim=True)
        # KL Divergence: D_KL( p(y|x) || p(y) )
        kl_div = part * (torch.log(part + 1e-10) - torch.log(p_y + 1e-10))
        sum_kl = torch.sum(kl_div, dim=1)
        avg_kl = torch.mean(sum_kl).item()
        split_scores.append(np.exp(avg_kl))

    return np.mean(split_scores), np.std(split_scores)



def run_benchmark_experiment(
    mode, temperature, top_k, beam_width, beam_temperature=3.5, num_samples=1024
):
    """
    Executes an isolated evaluation pass using LeNet predictions for scoring.
    """
    image_generator = INRImageGenerator(
        device=device,
        processor=processor,
        h=28,
        w=28,
        beam_width=beam_width,
        beam_temperature=beam_temperature,
        top_k=top_k,
        temperature=temperature,
    )

    is_batch_size = 64
    all_probabilities = []

    for _ in range(0, num_samples, is_batch_size):
        current_batch_size = min(is_batch_size, num_samples - _)

        fake_images = image_generator.generate_images(
            gpt_model=model,
            num_samples=current_batch_size,
            batch_loader_sample=reference_batch,
            mode=mode,
        )
        fake_images = fake_images.float()
        if fake_images.max() > 1.0:
            fake_images = fake_images / 255.0
        # Preprocessing safety: Ensure images are shaped [B, 1, 28, 28] and normalized to [0, 1] or [-1, 1]
        if fake_images.shape[1] != 1:
            fake_images = fake_images.mean(dim=1, keepdim=True) # Fallback to grayscale if channel mismatch

        with torch.no_grad():
            logits = lenet_model(fake_images)
            probs = F.softmax(logits, dim=1)
            all_probabilities.append(probs)

    # Concatenate all generated pool probabilities
    all_probabilities = torch.cat(all_probabilities, dim=0)
    
    # Compute the final custom LeNet Score
    ls_mean, ls_std = calculate_lenet_score(all_probabilities, splits=10)
    return ls_mean, ls_std

# %% [markdown]
# ### 4. Run the Grid Search Experiment Matrix

# %%
experiment_grid = [
    {"mode": "argmax", "temperature": 1.0, "top_k": 1, "beam_width": 1},
    {"mode": "sampling", "temperature": 1.0, "top_k": 9, "beam_width": 1},
    {"mode": "sampling", "temperature": 1.1, "top_k": 9, "beam_width": 1},
    {"mode": "sampling", "temperature": 1.25, "top_k": 15, "beam_width": 1},
    {"mode": "sampling", "temperature": 1.25, "top_k": 25, "beam_width": 1},
    {"mode": "sampling", "temperature": 1.35, "top_k": 50, "beam_width": 1},
    {"mode": "sampling", "temperature": 1.5, "top_k": 60, "beam_width": 1},
    {"mode": "sampling", "temperature": 1.7, "top_k": 70, "beam_width": 1},
    {"mode": "sampling", "temperature": 2.0, "top_k": 90, "beam_width": 1},
    {"mode": "sampling", "temperature": 2.2, "top_k": 100, "beam_width": 1},
]

results_registry = []

print("🔬 Starting Hyperparameter Search Matrix Evaluation (LeNet Engine)...\n")
print(f"{'MODE':<12} | {'TEMP':<5} | {'TOP-K':<5} | {'BEAM-TEMP':<9} | {'BEAM-W':<6} | {'LENET SCORE':<18}")
print("-" * 69)

for exp in experiment_grid:
    try:
        mean, std = run_benchmark_experiment(
            mode=exp["mode"],
            temperature=exp["temperature"],
            top_k=exp["top_k"],
            beam_width=exp["beam_width"],
            num_samples=1024,
        )

        result_payload = {
            "Generation Mode": exp["mode"],
            "Temperature": exp["temperature"],
            "Top-K": exp["top_k"],
            "Beam Width": exp["beam_width"],
            "LS Mean": round(mean, 4),
            "LS Std": round(std, 4),
            "Display String": f"{mean:.4f} ± {std:.4f}",
        }

        results_registry.append(result_payload)

        print(f"{exp['mode']:<12} | {exp['temperature']:<5} | {exp['top_k']:<5} | {3.5:<9} | {exp['beam_width']:<6} | {result_payload['Display String']:<18}")

    except Exception as e:
        print(f"❌ Failed combo {exp['mode']} | Temp {exp['temperature']}: {str(e)}")

# %% [markdown]
# ### 5. Final Leaderboard Analysis & Sorting

# %%
print("\n--- Final Sorting Matrix Results ---")
df = pd.DataFrame(results_registry)

leaderboard = df.sort_values(by="LS Mean", ascending=False).reset_index(drop=True)
print(leaderboard[["Generation Mode", "Temperature", "Top-K", "Beam Width", "Display String"]])